In [ ]:
from pathlib import Path
from ai_news_digest.config import Settings
from ai_news_digest.pipeline import collect_articles, rank_and_select, summarize_and_build, _get_llm
from datetime import datetime

# 🧠 AI News Digest — Pipeline

Genera un resumen mensual de noticias de IA listo para publicar en Medium.

**Pasos:**
1. **Recolección** — ArticlesAPI + TechCrunch scraping (~200 artículos)
2. **Ranking** — Score LLM 1-10 en batches de 20, selección top N
3. **Resúmenes** — LLM genera titular + resumen narrativo en español por artículo
4. **Guardado** — `.txt` (Markdown) + `.csv` (metadata)

> Ejecutar las celdas en orden. Configurar los parámetros en la celda siguiente antes de empezar.

In [ ]:
# Parámetros de corrida
DAYS = 30
TOP_N = 20 ##15
#ENSURE_SOURCES = [] 
ENSURE_SOURCES = ['Kdnuggets.com']
MONTH_NAME = "marzo_26"
LANG = "es"

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_PATH = "outputs/Highlights_AI" + "_"+ MONTH_NAME + "_" + ts +".txt"
print(f"Output path: {OUT_PATH}")

settings = Settings()

## ⚙️ Configuración

Ajustar estos parámetros antes de cada corrida:

| Parámetro | Descripción |
|-----------|-------------|
| `DAYS` | Cuántos días hacia atrás buscar artículos |
| `TOP_N` | Cantidad de artículos a incluir en el resumen final |
| `ENSURE_SOURCES` | Fuentes que deben aparecer al menos una vez (puede ser lista vacía `[]`) |
| `MONTH_NAME` | Nombre del mes — se usa en el título del artículo y el nombre del archivo |
| `LANG` | Idioma de los resúmenes (`"es"` para español) |

In [ ]:
# Cliente LLM compartido — acumula tokens entre ranking y summary
llm_client = _get_llm(settings)
print(f"LLM client: {type(llm_client).__name__ if llm_client else 'None'}")

## 📥 Paso 1 — Recolección de artículos

Recolecta artículos de **NewsAPI** (EN + ES) y **TechCrunch** (scraping).
El resultado se deduplica por URL y se guarda en `df_all`.

> Tarda ~2-3 minutos dependiendo de la conexión.

In [ ]:
print("🔎 [1/4] Recolectando artículos...")
df_all = collect_articles(settings, days=DAYS)
print(f"   → Artículos recolectados: {len(df_all)}")
if df_all.empty:
        print("   ⚠️  No se encontraron artículos.")

df_all.head(5)

In [ ]:
df_all.fuente.value_counts()

In [ ]:
df_all.to_csv("outputs/all_articles_mar_26_v2.csv", index=False)

## 📊 Paso 2 — Ranking por relevancia

Puntúa cada artículo del 1 al 10 usando el LLM en **batches de 20** para minimizar requests.
Luego filtra por diversidad de fuentes y selecciona el **top N**.

> Con ~200 artículos hace ~10 llamadas al LLM. Tarda ~1-2 minutos.

In [ ]:
import pandas as pd
df_all = pd.read_csv("outputs/all_articles_mar_26.csv")
df_all.fuente.value_counts()
df_all.head(5)

In [ ]:
print("📊 [2/4] Rankeando artículos por relevancia...")
df_top = rank_and_select(df_all, settings, top_n=TOP_N, ensure_source=ENSURE_SOURCES, llm_client=llm_client)
print(f"   → Seleccionados top {len(df_top)} artículos")
if llm_client and hasattr(llm_client, "token_summary"):
    print("   ", llm_client.token_summary())

In [ ]:
df_top.fuente.value_counts()

In [ ]:
df_top.to_csv("outputs/top_ranked_articles_mar_26_openAI.csv", index=False)

## 📝 Paso 3 — Generación de resúmenes

Para cada artículo seleccionado el LLM genera un **titular profesional** y un **resumen narrativo de 220-300 palabras en español**.
También extrae la imagen OG de cada URL.

> Con 20 artículos y 8s de delay entre llamadas tarda ~3-4 minutos.

In [ ]:
# import pandas as pd
# df_top = pd.read_csv("outputs/top_ranked_articles_mar_26.csv")
# print(df_top.fuente.value_counts())
# df_top.head(5)

In [ ]:
print("📝 [3/4] Generando resúmenes y armando artículo...")
article, df_res = summarize_and_build(df_top, settings, month_name=MONTH_NAME, lang=LANG, llm_client=llm_client)
print("   → Resúmenes generados")
if llm_client and hasattr(llm_client, "token_summary"):
    print("   ", llm_client.token_summary())

In [ ]:
df_res.resumen

In [ ]:
article

In [ ]:
print("💾 [4/4] Guardando resultados...")
Path(OUT_PATH).write_text(article, encoding="utf-8")
csv_path = Path(OUT_PATH).with_suffix(".csv")
csv_path.write_text(df_res.to_csv(index=False), encoding="utf-8")
print(f"   → Guardado {OUT_PATH} y {csv_path.name}")

print("✅ Pipeline finalizado con éxito.")

## 💾 Paso 4 — Guardado y resumen final

Guarda el artículo en `outputs/` como `.txt` (Markdown listo para Medium) y `.csv` (metadata).
Al final se muestra el total de tokens usados en el pipeline.

In [ ]:
# Total de tokens usados en todo el pipeline
if llm_client and hasattr(llm_client, "token_summary"):
    print("📊 Token usage total del pipeline:")
    print("  ", llm_client.token_summary())
else:
    print("No LLM client disponible — no hay tokens que reportar.")